##### **loading the sheet as excel**

In [51]:
import pandas as pd
import re
from datetime import datetime

# Google Sheet URL
url = "https://docs.google.com/spreadsheets/d/17K-ty-3SlAfn5NqSKfWVSJNchdv5tHrpq3dMUV8e6_Q/edit?usp=sharing"

# Convert to Excel export
excel_url = url.split("/edit")[0] + "/export?format=xlsx"

# Load all sheets
all_sheets = pd.read_excel(excel_url, sheet_name=None, engine="openpyxl")


##### **month filtering**

In [52]:
month_map = {
    "jan": "jan", "january": "jan",
    "feb": "feb", "february": "feb", "febrauray": "feb", "febraury": "feb",
    "mar": "mar", "march": "mar",
    "apr": "apr", "april": "apr",
    "may": "may",
    "jun": "jun", "june": "jun",
    "jul": "jul", "july": "jul",
    "aug": "aug", "august": "aug",
    "sep": "sep", "september": "sep",
    "oct": "oct", "october": "oct",
    "nov": "nov", "november": "nov",
    "dec": "dec", "december": "dec"
}

def is_valid_year(y):
    y = int(y)
    if 2020 <= y <= 2030:
        return True
    if 20 <= y <= 30:  # short format → 2020–2030
        return True
    return False


def is_valid_month_year_sheet(name):
    name = name.strip().lower()

    # Only allow letters + space (strict)
    if not re.fullmatch(r"[a-z0-9\s]+", name):
        return False

    parts = name.split()

    if len(parts) != 2:
        return False

    part1, part2 = parts

    # Case 1: month year
    if part1 in month_map and part2.isdigit() and is_valid_year(part2):
        return True

    # Case 2: year month
    if part2 in month_map and part1.isdigit() and is_valid_year(part1):
        return True

    return False


# Apply filter
monthly_sheets_filtered = {
    name: df for name, df in all_sheets.items()
    if is_valid_month_year_sheet(name)
}

In [53]:
for sheet_name, df in monthly_sheets_filtered.items():
    print(f"\n📄 Sheet: {sheet_name}")
    print(list(df.columns))


📄 Sheet: MARCH 2026
['DATE', 'AWB NO.', 'SENDER NAME', 'ADDRESS', 'CONTACT NO', 'RECEIVER NAME', 'CONTACT NO.', 'ADDRESS.1', 'PIN CODE', 'DESTINATION', 'WEIGHT\n IN KG', 'Amount', 'MODE', 'CREDIT OR CASH', 'Unnamed: 14', 'STATUS ', 'Unnamed: 16']

📄 Sheet: FEB 2026
['DATE', 'AWB NO.', 'SENDER NAME', 'ADDRESS', 'CONTACT NO', 'RECEIVER NAME', 'CONTACT NO.', 'ADDRESS.1', 'PIN CODE', 'DESTINATION', 'WEIGHT\n IN KG', 'Amount', 'MODE', 'CREDIT OR CASH', 'Unnamed: 14', 'STATUS ', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 5

##### **checking sheet data formated**

In [54]:
for sht, dff in monthly_sheets.items():
    if sht == "AUG-2024":
        print(f"\nSheet: {sht}")

        print("\nColumns:")
        print(", ".join(dff.columns.astype(str)))

        print("\nSample Data:\n")
        print(dff.head(10).to_string(index=False))


Sheet: AUG-2024

Columns:
Unnamed: 0, DATE, AWB NO., SENDER NAME, ADDRESS, CONTECT NO, RECEIVER NAME, CONTACT NO., ADDRESS.1, PIN CODE, DESTINATION, WEIGHT
 IN KG, credit, AMOUNT
IN INR, MODE, REMARKS, STATUS , DELIVERY DATE, REFERENCE, CUSTOMER ID 

Sample Data:

 Unnamed: 0       DATE      AWB NO.                SENDER NAME      ADDRESS CONTECT NO              RECEIVER NAME CONTACT NO.                                                     ADDRESS.1  PIN CODE DESTINATION WEIGHT\n IN KG credit  AMOUNT\nIN INR    MODE REMARKS   STATUS   DELIVERY DATE  REFERENCE  CUSTOMER ID 
        NaN 2024-08-01 100156770683                 sharma ji    sewak park 9871773762    ms.trans world courier,  6395487767                         a-5shanker market ,gandi park aligarh  202001.0     aligarh            0.1  cash             50.0 SURFACE     NaN DELIVERED            NaN        NaN           NaN
        NaN 2024-08-01 100156770692                 disha tech  dwarka mor         NaN jan aushedi medical

##### **making each sheet standard wise**

In [55]:

cleaned_data = []

for sheet_name, df in monthly_sheets_filtered.items():
    df = df.copy()

    # Rename first column to DATE
    df.rename(columns={df.columns[0]: "DATE"}, inplace=True)

    # Convert date
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")


    df.columns = df.columns.str.strip()


    # check awb
    if "AWB NO." not in df.columns:
        continue


    # Tender column detection
    tender_col = None
    if "MODE" in df.columns:
        tender_col = "MODE"
    elif "BILLING DETAILS" in df.columns:
        tender_col = "BILLING DETAILS"

    # Payment column
    if "CREDIT OR CASH" in df.columns:
        df["CREDIT OR CASH"] = df["CREDIT OR CASH"].astype(str).str.upper().str.strip()

    # Amount cleanup
    if "Amount" in df.columns:
        df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

    df["Sheet"] = sheet_name

    cleaned_data.append(df)

# Combine all monthly sheets
data = pd.concat(cleaned_data, ignore_index=True)

##### **date wise packet count sorted**

In [56]:
date_wise_all = []

for sheet_name, df in monthly_sheets_filtered.items():
    df = df.copy()

    # Clean column names
    df.columns = df.columns.astype(str).str.strip().str.upper()

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Select date column
    if "DATE" in df.columns:
        date_col = "DATE"
    elif "AHU" in df.columns:
        date_col = "AHU"
    else:
        date_col = df.columns[0]

    # Convert to datetime
    df["DATE"] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)

    # Drop invalid dates
    df = df[df["DATE"].notna()]

    # Ensure AWB exists
    if "AWB NO." not in df.columns:
        continue

    # Count packets
    temp = df.groupby("DATE")["AWB NO."].count().reset_index(name="Packet Count")
    temp["Sheet"] = sheet_name

    date_wise_all.append(temp)

# Combine all sheets
date_wise_result = pd.concat(date_wise_all, ignore_index=True)

# ✅ Sort: first by Sheet, then highest packet count
sorted_result = date_wise_result.sort_values(
    ["Sheet", "Packet Count"],
    ascending=[True, False]
)

print(sorted_result)

          DATE  Packet Count           Sheet
252 2025-04-07            37      APRIL 2025
271 2025-04-29            30      APRIL 2025
270 2025-04-28            29      APRIL 2025
259 2025-04-15            25      APRIL 2025
269 2025-04-26            25      APRIL 2025
..         ...           ...             ...
138 2025-09-24            20  SEPTEMBER 2025
126 2025-09-10            17  SEPTEMBER 2025
122 2025-09-05            16  SEPTEMBER 2025
133 2025-09-18            16  SEPTEMBER 2025
142 2025-09-29            13  SEPTEMBER 2025

[352 rows x 3 columns]


##### **packets per tender count sorted**

In [57]:
tender_all = []

for sheet_name, df in monthly_sheets_filtered.items():
    df = df.copy()

    # Clean column names
    df.columns = df.columns.astype(str).str.strip().str.upper()

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Ensure AWB exists
    if "AWB NO." not in df.columns:
        continue

    # ✅ Detect sender column (tender)
    if "SENDER NAME" in df.columns:
        sender_col = "SENDER NAME"
    elif "MODE" in df.columns:
        sender_col = "MODE"
    elif "BILLING DETAILS" in df.columns:
        sender_col = "BILLING DETAILS"
    else:
        continue

    # Group by sender → count packets
    temp = (
        df.groupby(sender_col)["AWB NO."]
        .count()
        .reset_index(name="Packet Count")
    )

    temp["Sheet"] = sheet_name

    tender_all.append(temp)

# Combine all sheets
tender_result = pd.concat(tender_all, ignore_index=True)

# Sort within each month (highest → lowest)
tender_result = tender_result.sort_values(
    ["Sheet", "Packet Count"],
    ascending=[True, False]
)

# Display month-wise
for sheet, group in tender_result.groupby("Sheet"):
    print(f"\n📦 {sheet} (Sender-wise Packets)\n")

    # Drop Sheet column for display
    display_df = group[["SENDER NAME", "Packet Count"]]

    print(display_df.to_string(index=False))


📦 APRIL 2025 (Sender-wise Packets)

                                                    SENDER NAME  Packet Count
                            ELWINT LOGISTICS SOLUTIONS PVT. LTD            97
                                                   UJJIVAN BANK            76
                                                      ABHIKSHA             42
                                       ACCREDIUM CERTIFICATIONS            30
                                                  ANANYA SAXENA            27
                                                  RAKHI KUMARI             12
                                          LEXA FILTERS& FABRICS            10
                                                     TATTWAMASI            10
                                                         K.F.H              9
                                                        KRISHWI             8
                                                      EASY LIFE             7
                           

##### **mode wise packets booked per month sorted**

In [58]:
mode_all = []

for sheet_name, df in monthly_sheets_filtered.items():
    df = df.copy()

    # Clean column names
    df.columns = df.columns.astype(str).str.strip().str.upper()

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Ensure required columns exist
    if "AWB NO." not in df.columns or "MODE" not in df.columns:
        continue

    # Normalize MODE values (optional but recommended)
    df["MODE"] = df["MODE"].astype(str).str.upper().str.strip()

    # Group by MODE
    temp = (
        df.groupby("MODE")["AWB NO."]
        .count()
        .reset_index(name="Packet Count")
    )

    temp["Sheet"] = sheet_name
    mode_all.append(temp)

# Combine all sheets
mode_result = pd.concat(mode_all, ignore_index=True)

# Sort highest → lowest within each month
mode_result = mode_result.sort_values(
    ["Sheet", "Packet Count"],
    ascending=[True, False]
)

# ✅ Display month-wise (clean format)
for sheet, group in mode_result.groupby("Sheet"):
    print(f"\n🚚 {sheet} (Mode-wise Packets)\n")
    print(group[["MODE", "Packet Count"]].to_string(index=False))


🚚 APRIL 2025 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           376
    AIR            58
    NAN             1

🚚 AUGUST 2025 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           515
    AIR            94
    NAN             1

🚚 DECEMBER 2025 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           718
    AIR           158

🚚 FEB 2026 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           581
    AIR           109

🚚 JAN 2026 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           740
    AIR           141

🚚 JULY 2025 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           519
    AIR           107
    NAN             0

🚚 JUNE 2025 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           406
    AIR            73
    NAN             2

🚚 MARCH 2026 (Mode-wise Packets)

   MODE  Packet Count
SURFACE           385
    AIR            53
    NAN             1

🚚 MAY 2025  (Mode-wise Packets)

   MODE  Packet Count
SURFACE           482
    AI

##### **How much payment received and how much credit**

In [59]:
for sheet_name, df in monthly_sheets_filtered.items():
    df = df.copy()

    # Clean column names
    df.columns = df.columns.astype(str).str.strip().str.upper()

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Ensure required columns exist
    required_cols = ["CREDIT OR CASH", "AMOUNT", "SENDER NAME"]
    if not all(col in df.columns for col in required_cols):
        continue

    # Normalize values
    df["CREDIT OR CASH"] = df["CREDIT OR CASH"].astype(str).str.upper().str.strip()
    df["AMOUNT"] = df["AMOUNT"].astype(str).str.strip()

    # Split numeric and 'monthly'
    df["AMOUNT_NUM"] = pd.to_numeric(df["AMOUNT"], errors="coerce")

    # =========================================================
    # ✅ CASH & UPI SUM
    # =========================================================
    cash_total = df[df["CREDIT OR CASH"] == "CASH"]["AMOUNT_NUM"].sum()
    upi_total = df[df["CREDIT OR CASH"] == "UPI"]["AMOUNT_NUM"].sum()

    # =========================================================
    # ✅ CREDIT NUMERIC SUM
    # =========================================================
    credit_numeric = df[df["CREDIT OR CASH"] == "CREDIT"]
    credit_total = credit_numeric["AMOUNT_NUM"].sum()

    # =========================================================
    # ✅ CREDIT MONTHLY COUNT PER SENDER
    # =========================================================
    credit_monthly = df[
        (df["CREDIT OR CASH"] == "CREDIT") &
        (df["AMOUNT"].str.lower() == "monthly")
    ]

    monthly_sender_count = (
        credit_monthly.groupby("SENDER NAME")["AMOUNT"]
        .count()
        .reset_index(name="Monthly Count")
        .sort_values("Monthly Count", ascending=False)
    )

    # =========================================================
    # ✅ DISPLAY
    # =========================================================
    print(f"\n💰 {sheet_name}\n")

    print(f"Cash Booking   : ₹ {int(cash_total) if pd.notna(cash_total) else 0}")
    print(f"UPI Booking    : ₹ {int(upi_total) if pd.notna(upi_total) else 0}")
    print(f"Credit Booking : ₹ {int(credit_total) if pd.notna(credit_total) else 0}")

    print("\n📊 Monthly Credit (per Sender):\n")

    if not monthly_sender_count.empty:
        print(monthly_sender_count.to_string(index=False))
    else:
        print("No monthly credit entries")


💰 MARCH 2026

Cash Booking   : ₹ 150
UPI Booking    : ₹ 2030
Credit Booking : ₹ 0

📊 Monthly Credit (per Sender):

                        SENDER NAME  Monthly Count
                      PROTEIN SCOOP             85
                      UJJIVAN  BANK             56
                        VIHAAN ENT.             39
               KRISHWI ENTERPRISESS             33
           ACCREDIUM CERTIFICATIONS             23
                         GAME FIXIT             18
                        LEXA FILTER             16
 Elwint Logistics Solutions Pvt Ltd             12
                           ABHIKSHA             10
 ELVINT LOGISTICS SOLUTIONS PVT LTD              9
                            RATNAM               8
                      RAKHI KUMARI               8
                        SUMIT MEHTA              7
                       RAKHI KUMARI              7
PYOLI-TRANS LOGISTICS INDIA PVT LTD              6
                       Rakhi Kumari              5
                 